In [1]:
import scipy
import numpy as np
import pandas as pd

import torch
import torchaudio
import torchaudio.transforms as T
import sys
import torch.optim as optim
from torch.optim.lr_scheduler import CyclicLR
from speechbrain.inference.speaker import EncoderClassifier
from transformers import AutoFeatureExtractor, AutoModel

sys.path.append('/om2/user/salavill/misc/voice-speech-metamers/')
from utils import *
from learner import Learner
from tokenizer import Tokenizer
from decoder import Speech_Decoder_Linear, Speaker_Decoder_Linear
from encoder import Speaker_Encoder, Speech_Encoder, Joint_Encoder

In [2]:
sr = 16000

# Load in ECAPA - Voice model

In [4]:
ecapa_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")
# target = model.encode_batch(signal)[0]

# Load in Whisper - Speech Model

In [8]:
# load in model 
whisper_feature_extractor = AutoFeatureExtractor.from_pretrained("openai/whisper-base")
whisper_encoder = AutoModel.from_pretrained("openai/whisper-base")#, cache_dir=cache_dir)
decoder_input_ids = torch.tensor([[1, 1]]) * whisper_encoder.config.decoder_start_token_id
whisper_encoder.eval()

def run_whisper(input, noise=False):
    """
    runs the whisper model when given audio input
    """
    input = whisper_feature_extractor(input.detach().cpu(), sampling_rate=sr, return_tensors="pt").input_features
    if noise:
        input = input.clone().requires_grad_()
    output = whisper_encoder(input, decoder_input_ids=decoder_input_ids)
    return output.encoder_last_hidden_state.mean(1)

# target = run_model(signal)


# Load in Joint Model

In [3]:


# load in model 
config_path = "../config.yaml"

# Load config file
config = load_yaml_config(config_path)


#define a tokenizer for the vocabulary
tokenizer = Tokenizer(**config.text)

speaker_encoder = Speaker_Encoder(config.encoder.model_cache)
speech_encoder = Speech_Encoder(config.encoder.model_cache)

#define joint encoder
saganet = Joint_Encoder(config.saganet.d_model,
                        config.saganet.num_head,
                        config.saganet.dim_feedforward,
                        config.saganet.num_layers)

#define decoders
speech_decoder = Speech_Decoder_Linear()
speaker_decoder = Speaker_Decoder_Linear()


checkpoint = "/om2/user/annesyab/SLP_Project_2024/saganet/saganet_d-704_atthead-8/best42-val_loss0.64.ckpt"
saganet = Learner.load_from_checkpoint(checkpoint_path=checkpoint,
                                                config=config, 
                                                tokenizer=tokenizer,
                                                speech_encoder=speech_encoder,
                                                speaker_encoder=speaker_encoder,
                                                joint_encoder=saganet,
                                                speech_decoder=speech_decoder,
                                                speaker_decoder = speaker_decoder,)

print('Loaded in joint model')

# Get target embedding by running signal through model
# target, _ = model(signal)

Loaded in joint model


In [27]:
signal.shape

(115200,)

In [8]:
saganet({'input_values': torch.from_numpy(signal).unsqueeze(0)})

(tensor([[[-1.7749e+00, -1.0650e-01,  1.9538e+00,  ...,  5.7962e-01,
           -7.9778e-01,  1.7537e-01],
          [-1.7750e+00, -1.2060e+00,  1.2682e-01,  ...,  1.7012e+00,
           -2.3599e-01, -6.3720e-01],
          [-3.0227e-01,  1.1780e-01, -2.1703e-01,  ...,  3.0039e-01,
            2.8216e-02,  3.8157e-01],
          ...,
          [-2.3307e-01,  7.2763e-01, -8.9949e-01,  ...,  8.8728e-02,
            4.4962e-01,  6.1656e-02],
          [ 1.4706e-01,  1.0263e+00, -1.8178e+00,  ..., -7.4496e-01,
            1.0273e+00, -3.4392e-01],
          [-1.7430e+00, -5.8965e-02,  1.6100e-01,  ..., -4.2017e-01,
           -5.6464e-01, -1.3246e-03]]], device='cuda:0',
        grad_fn=<NativeLayerNormBackward0>),
 ['t h e   p a s e   f e w   y e a r s   h a v e w i n e s e t d   i m p o r t a n n t   g r o v e   f o r   t t h e   a c e m y'])

# Create Joint Babble Data

In [7]:
def combine_signal_and_noise(signal, noise, snr, mean_subtract=True):
    '''
    Adds noise to signal with the specified signal-to-noise ratio (snr).
    If snr is finite, the noise waveform is rescaled and added to the
    signal waveform. If snr is positive infinity, returned waveform is
    equal to the signal waveform. If snr is negative inifinity, returned
    waveform is equal to the noise waveform.
    
    Args
    ----
    signal (np.ndarray): signal waveform
    noise (np.ndarray): noise waveform
    snr (float): signal-to-noise ratio in dB
    mean_subtract (bool): if True, signal and noise are first de-meaned
        (mean_subtract=True is important for accurate snr computation)
    
    Returns
    -------
    signal_and_noise (np.ndarray) signal in noise waveform
    '''
    rms = lambda stim: np.sqrt(np.mean(stim * stim))

    if mean_subtract:
        signal = signal - np.mean(signal)
        noise = noise - np.mean(noise)        
    if np.isinf(snr) and snr > 0:
        signal_and_noise = signal
    elif np.isinf(snr) and snr < 0:
        signal_and_noise = noise
    else:
        rms_noise_scaling = rms(signal) / (rms(noise) * np.power(10, snr / 20))
        signal_and_noise = signal + rms_noise_scaling * noise
    return signal_and_noise

In [4]:
import glob
backgrounds = glob.glob('/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/background_cv08talkerbabble/*.wav')

In [12]:
pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/background_cv08talkerbabble/manifest.pdpkl')

'/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/background_cv08talkerbabble/manifest.pdpkl'

In [13]:
pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/background_cv08talkerbabble/manifest.pdpkl')

,clip_dur_in_s,clip_end_in_s,clip_start_in_s,raw_clip_dur_in_s,raw_clip_end_in_s,raw_clip_start_in_s,raw_src_fn,raw_total_file_duration_in_s,sr,src_fn,total_file_duration_in_s
0,3.0,3.0,0.0,"[0.08018750000000008, 0.02012499999999995, 0.2...","[1.964625, 1.787625, 1.770125, 3.8514375, 2.08...","[1.8844375, 1.7675, 1.548875, 3.3499375, 1.502...",[/om2/data/public/mozilla-CommonVoice-9.0/cv-c...,"[5.19825, 4.716, 3.504, 8.088, 6.588, 6.456, 3...",44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0
1,3.0,3.0,0.0,"[0.060125000000000206, 0.5423749999999998, 0.0...","[3.668, 2.5913125, 3.0045, 3.18725, 6.5933125,...","[3.607875, 2.0489375, 2.944375, 3.1070625, 5.9...",[/om2/data/public/mozilla-CommonVoice-9.0/cv-c...,"[5.496, 4.584, 5.352, 8.784, 9.936, 6.984, 6.4...",44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0
2,3.0,3.0,0.0,"[0.12068749999999984, 0.08018749999999963, 0.2...","[2.1123125, 2.5858125, 2.03125, 3.0350625, 1.9...","[1.991625, 2.505625, 1.7496875, 2.87425, 1.884...",[/om2/data/public/mozilla-CommonVoice-9.0/cv-c...,"[4.14, 5.328, 3.636, 4.86, 4.656, 4.632, 7.944...",44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0
3,3.0,3.0,0.0,"[0.10031250000000025, 0.3213124999999999, 0.06...","[2.2865, 1.8473125, 3.1065, 1.8871875, 2.26506...","[2.1861875, 1.526, 3.046375, 1.545875, 2.12475...",[/om2/data/public/mozilla-CommonVoice-9.0/cv-c...,"[4.248, 5.064, 7.632, 4.2, 5.376, 4.464, 6.36,...",44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0
4,3.0,3.0,0.0,"[0.10018749999999987, 0.16037500000000016, 0.5...","[2.0846875, 1.904875, 4.2888125, 5.853, 2.7308...","[1.9845, 1.7445, 3.7476875, 5.572375, 2.128437...",[/om2/data/public/mozilla-CommonVoice-9.0/cv-c...,"[7.08, 4.656, 5.868, 9.024, 5.064, 8.208, 8.83...",44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0
...,...,...,...,...,...,...,...,...,...,...,...
395,3.0,3.0,0.0,"[0.22062500000000007, 0.08012499999999978, 0.0...","[3.249, 4.44825, 7.7695, 2.5071875, 2.2865, 2....","[3.028375, 4.368125, 7.7494375, 1.8453125, 2.1...",[/om2/data/public/mozilla-CommonVoice-9.0/cv-c...,"[7.224, 6.456, 9.828, 4.176, 4.248, 4.584, 4.5...",44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0
396,3.0,3.0,0.0,"[0.04012500000000019, 0.30031249999999954, 0.2...","[3.070625, 6.1264375, 1.9021875, 6.4935, 4.008...","[3.0305, 5.826125, 1.6619375, 6.2730625, 3.968...",[/om2/data/public/mozilla-CommonVoice-9.0/cv-c...,"[5.796, 7.632, 10.176, 9.624, 5.976, 3.504, 3....",44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0
397,3.0,3.0,0.0,"[0.4419999999999997, 0.1001875000000001, 0.140...","[3.6965, 3.26775, 1.6426875, 3.0886875, 2.0865...","[3.2545, 3.1675625, 1.5024375, 2.848, 1.946125...",[/om2/data/public/mozilla-CommonVoice-9.0/cv-c...,"[5.4, 6.72, 7.296, 7.104, 3.816, 3.996, 6.096,...",44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0
398,3.0,3.0,0.0,"[0.18081250000000004, 0.1403749999999997, 0.14...","[2.4515625, 2.426125, 2.2065625, 2.424375, 1.8...","[2.27075, 2.28575, 2.066125, 2.0636875, 1.7673...",[/om2/data/public/mozilla-CommonVoice-9.0/cv-c...,"[5.088, 4.776, 4.068, 4.464, 4.824, 4.776, 7.5...",44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0


In [9]:
pd.read_csv('/om2/user/gelbanna/commonvoice_data_curated.csv').query('split == "test" and total_file_duration_in_s > 2').iloc[0]

Unnamed: 0                                                            1479941
client_id                   372293e65cdab88771e028a4351651ab2eff64438ddafc...
sr                                                                      48000
wav_path                    /om2/data/public/mozilla-CommonVoice-9.0/cv-co...
total_file_duration_in_s                                                  7.2
gender                                                                   male
sentence                    the past few years have witnessed important gr...
speaker_int                                                                 0
split                                                                    test
Name: 345, dtype: object

In [10]:
pd.read_csv('/om2/user/gelbanna/commonvoice_data_curated.csv').query('split == "test" and total_file_duration_in_s > 2').iloc[0].sentence

'the past few years have witnessed important growth for the academy'

In [11]:
sig_path = pd.read_csv('/om2/user/gelbanna/commonvoice_data_curated.csv').query('split == "test" and total_file_duration_in_s > 2').iloc[0].wav_path
noise_path = backgrounds[0]
sr = 16000

In [6]:

signal, fs = torchaudio.load(sig_path)
if fs != sr:
    # make sure to resample to appropriate frequency
    print('resampling audio')
    resampler = T.Resample(fs, sr, dtype=signal.dtype)
    signal = resampler(signal)

if len(signal.shape)>1:
    # Reshape signal as necessary
    signal = torch.squeeze(signal)

signal = signal.numpy()

resampling audio


In [37]:
noise.shape, signal.shape

((48000,), (115200,))

In [36]:
noise, fs = torchaudio.load(noise_path)
if fs != sr:
    # make sure to resample to appropriate frequency
    print('resampling audio')
    resampler = T.Resample(fs, sr, dtype=noise.dtype)
    noise = resampler(noise)

if len(noise.shape)>1:
    # Reshape signal as necessary
    noise = torch.squeeze(noise)
noise[:signal.shape[0]]

noise = noise.numpy()

signal = combine_signal_and_noise(signal, noise, 25, mean_subtract=True)

resampling audio


ValueError: operands could not be broadcast together with shapes (115200,) (48000,) 

In [19]:
run_whisper(torch.from_numpy(signal))

tensor([[-1.9141e-02,  6.5450e-02,  3.0290e-01, -7.8746e-01,  1.0538e-01,
          1.2610e+00, -3.3319e-01, -4.1058e-02, -1.5894e-01,  1.6592e-02,
         -2.9071e-01,  1.0708e-01, -2.5317e-01,  5.7739e-01,  6.1053e-01,
         -2.0196e-01,  1.9186e-01,  1.4000e-01,  3.2418e-01,  3.3653e-02,
         -1.7498e-01, -8.3087e-02,  1.5311e-01,  1.3246e-01,  3.3202e-02,
          1.4946e+00, -3.4409e-01,  2.7161e-01, -3.4331e-01, -1.7063e+00,
         -2.5244e-01,  6.0569e-01, -2.5622e-01, -2.4654e-01,  1.2582e-01,
          8.9018e-02, -3.8767e-02,  6.2064e-02,  1.3614e-01,  1.5439e-01,
          7.1374e-02,  1.1099e+00,  5.6705e-01,  2.4700e-01, -3.5858e-02,
          3.4450e-01,  4.6595e-02, -3.2373e-01,  3.5717e-01,  1.9175e-01,
         -4.4550e-01, -6.9056e-02,  1.2117e-02, -1.0320e-01,  1.8658e-01,
          3.3638e-01,  1.3529e-01, -3.2069e-01, -5.9008e-02,  8.8172e-02,
         -7.0324e-02,  1.2123e+00, -1.2969e-01,  9.2169e-02,  1.4024e-01,
          4.2999e-01, -9.3146e-02,  2.

# Test Models